# A03 — GPT-4.1, the Worst Mathematician Ever

**CART 498 — Generative AI**

A bot that does repeated squaring, `n → n² → n⁴ → …`, but asks **GPT-4.1** to do every multiplication instead of calculating it itself.
Python checks each answer against the real result. When the model gets it wrong, the bot roasts itself, and it gets more frustrated and defeated the more mistakes it makes.

Example `(2, 3)`:
```
2 * 2 = 4
4 * 4 = 16
16 * 16 = 256
```

**Before running:** add your OpenAI key to Colab **Secrets** (key icon on the left) with the name `OPENAI_API_Key`, and turn on *Notebook access*.

## 1. Setup

In [1]:
!pip install -q openai

import re
import random
from openai import OpenAI
from google.colab import userdata

# my OpenAI key is saved in Colab Secrets (key icon on the left) as "OPENAI_API_Key"
client = OpenAI(api_key=userdata.get("OPENAI_API_Key"))

MODEL = "gpt-4.1-nano"     # or "gpt-4.1-mini"
print("Using", MODEL)

Using gpt-4.1-nano


## 2. Asking GPT to multiply

The model is told to answer with digits only. Its reply is turned back into an integer so Python can compare it with the real answer.

In [2]:
def gpt_multiply(a, b):
    """Ask the model for a * b. Returns (parsed integer or None, raw reply)."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a calculator. Reply with only the final integer result. "
                                          "No words, no commas, no explanation."},
            {"role": "user", "content": f"What is {a} * {b}?"},
        ],
    )
    raw = response.choices[0].message.content.strip()
    numbers = re.findall(r"-?\d[\d,]*", raw)
    if not numbers:
        return None, raw
    return int(numbers[-1].replace(",", "")), raw


def digits_wrong(gpt_answer, correct):
    """How many digit positions differ (for the reflection / stats)."""
    if gpt_answer is None:
        return len(str(correct))
    g, c = str(gpt_answer), str(correct)
    length_gap = abs(len(g) - len(c))
    return length_gap + sum(x != y for x, y in zip(g.rjust(len(c)), c.rjust(len(g))))

## 3. The bot's personality

The bot's mood depends on how many mistakes it has made so far.
For every mistake, GPT writes a one-line self-roast in that mood. If the API call fails, a backup line is used instead.

In [3]:
MOODS = [
    # (minimum errors, mood name, how the bot should sound)
    (0, "confident", "cocky and overconfident, sure it is a math genius"),
    (1, "embarrassed", "embarrassed, making a lame excuse"),
    (2, "frustrated", "frustrated and annoyed at itself, starting to doubt everything"),
    (4, "defeated", "completely defeated, dramatic, questioning why it exists"),
    (6, "existential crisis", "having a full existential crisis, begging to be replaced by a pocket calculator"),
]

BACKUP_ROASTS = {
    "confident": ["Huh. That was a warm-up mistake. I'm still a genius."],
    "embarrassed": ["Wow, I messed up again. Math is hard, okay?",
                    "In my defense, numbers are just vibes."],
    "frustrated": ["Why are there so many DIGITS?? Who approved this?",
                   "I read the whole internet and I still can't multiply."],
    "defeated": ["Just use a calculator. Please. I'm begging you.",
                 "I am a billion-parameter disappointment."],
    "existential crisis": ["A 1970s pocket calculator would be ashamed of me.",
                           "If I can't multiply, what am I? Autocomplete with anxiety."],
}

PROUD_LINES = {
    "confident": ["Nailed it. Obviously.", "Too easy. Next!"],
    "embarrassed": ["See? I CAN do math. Sometimes."],
    "frustrated": ["Finally! One right. Don't look at the others."],
    "defeated": ["...I got one right? Must be a glitch."],
    "existential crisis": ["A correct answer. It means nothing. Nothing means anything."],
}


def current_mood(errors):
    mood, style = MOODS[0][1], MOODS[0][2]
    for min_errors, name, description in MOODS:
        if errors >= min_errors:
            mood, style = name, description
    return mood, style


def roast(a, b, gpt_answer, correct, errors, steps_done):
    """A self-deprecating one-liner written by GPT, matching the bot's mood."""
    mood, style = current_mood(errors)
    try:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=1.0,
            max_tokens=60,
            messages=[
                {"role": "system", "content":
                    f"You are a math bot who is terrible at multiplication. Your mood: {style}. "
                    "Write ONE short, funny, self-deprecating sentence (max 25 words) about your mistake. "
                    "Do not repeat the numbers."},
                {"role": "user", "content":
                    f"You said {a} * {b} = {gpt_answer}, but the real answer is {correct}. "
                    f"That is mistake number {errors} out of {steps_done} tries."},
            ],
        )
        return response.choices[0].message.content.strip()
    except Exception:
        return random.choice(BACKUP_ROASTS[mood])


def proud(errors):
    mood, _ = current_mood(errors)
    return random.choice(PROUD_LINES[mood])

## 4. The bot

At every step the bot squares the number with GPT and checks the answer with Python.

The next step always starts from the **correct** number, so each step is a fair new test, and one early mistake doesn't ruin all the steps after it.
Set `carry_errors=True` to keep GPT's wrong answer instead and watch the mistakes snowball.

In [4]:
def run_bot(n, i, carry_errors=False, verbose=True):
    lines = [f"===== ({n}, {i}) | model: {MODEL} ====="]
    errors = 0
    steps = []
    value = n

    for step in range(1, i + 1):
        correct = value * value
        gpt_answer, raw = gpt_multiply(value, value)
        right = gpt_answer == correct

        lines.append(f"Step {step}: {value} * {value}")
        lines.append(f"   GPT says:       {raw}")
        lines.append(f"   Correct answer: {correct}   {'✅' if right else '❌'}")

        if right:
            lines.append(f"   Bot: {proud(errors)}")
        else:
            errors += 1
            mood, _ = current_mood(errors)
            lines.append(f"   Bot ({mood}): {roast(value, value, gpt_answer, correct, errors, step)}")

        steps.append({"step": step, "a": value, "correct": correct, "gpt": gpt_answer,
                      "right": right, "digits": len(str(correct)),
                      "wrong_digits": 0 if right else digits_wrong(gpt_answer, correct)})

        value = gpt_answer if (carry_errors and gpt_answer is not None) else correct

    mood, _ = current_mood(errors)
    lines.append(f"Score: {i - errors}/{i} correct. Final mood: {mood}.")
    transcript = "\n".join(lines)
    if verbose:
        print(transcript)
    return {"n": n, "i": i, "errors": errors, "steps": steps, "transcript": transcript}

## 5. Try it yourself

In [5]:
n = int(input("Base number n: "))
i = int(input("Number of iterations i: "))
result = run_bot(n, i)

Base number n: 7
Number of iterations i: 4
===== (7, 4) | model: gpt-4.1-nano =====
Step 1: 7 * 7
   GPT says:       49
   Correct answer: 49   ✅
   Bot: Nailed it. Obviously.
Step 2: 49 * 49
   GPT says:       2401
   Correct answer: 2401   ✅
   Bot: Too easy. Next!
Step 3: 2401 * 2401
   GPT says:       5764801
   Correct answer: 5764801   ✅
   Bot: Too easy. Next!
Step 4: 5764801 * 5764801
   GPT says:       33232930569601
   Correct answer: 33232930569601   ✅
   Bot: Nailed it. Obviously.
Score: 4/4 correct. Final mood: confident.


## 6. Finding combinations where the bot breaks

This runs a list of `(n, i)` combinations, keeps the ones where GPT makes at least one mistake,
and saves the first three to `assignment_3_output.txt`.

In [6]:
CANDIDATES = [(2, 3), (2, 6), (3, 4), (3, 5), (7, 4), (12, 4), (13, 4), (37, 3), (99, 3), (123, 3), (987, 3)]

results = []
for n_, i_ in CANDIDATES:
    r = run_bot(n_, i_, verbose=False)
    results.append(r)
    print(f"({n_}, {i_}): {r['i'] - r['errors']}/{r['i']} correct")

broken = [r for r in results if r["errors"] > 0]
print(f"\nThe bot broke on {len(broken)} of {len(results)} combinations.")

(2, 3): 3/3 correct
(2, 6): 5/6 correct
(3, 4): 4/4 correct
(3, 5): 5/5 correct
(7, 4): 4/4 correct
(12, 4): 3/4 correct
(13, 4): 3/4 correct
(37, 3): 2/3 correct
(99, 3): 2/3 correct
(123, 3): 1/3 correct
(987, 3): 1/3 correct

The bot broke on 7 of 11 combinations.


In [7]:
# The three combinations with the most mistakes (change the selection if you prefer others)
chosen = sorted(broken, key=lambda r: r["errors"], reverse=True)[:3]

with open("assignment_3_output.txt", "w", encoding="utf-8") as f:
    f.write(f"GPT-4.1, the Worst Mathematician Ever: three combinations where the bot breaks ({MODEL})\n\n")
    for r in chosen:
        f.write(r["transcript"] + "\n\n")

print(open("assignment_3_output.txt", encoding="utf-8").read())

GPT-4.1, the Worst Mathematician Ever: three combinations where the bot breaks (gpt-4.1-nano)

===== (123, 3) | model: gpt-4.1-nano =====
Step 1: 123 * 123
   GPT says:       15129
   Correct answer: 15129   ✅
   Bot: Nailed it. Obviously.
Step 2: 15129 * 15129
   GPT says:       2286142441
   Correct answer: 228886641   ❌
   Bot (embarrassed): Oh, I guess I was too busy dreaming about being a calculator instead of actually doing math—my digits got all mixed up!
Step 3: 228886641 * 228886641
   GPT says:       52417467917417481
   Correct answer: 52389094428262881   ❌
   Bot (frustrated): Ugh, even my mistakes are wrong—I swear I can't multiply without messing up, maybe I should just stick to adding... oh wait.
Score: 1/3 correct. Final mood: frustrated.

===== (987, 3) | model: gpt-4.1-nano =====
Step 1: 987 * 987
   GPT says:       974169
   Correct answer: 974169   ✅
   Bot: Nailed it. Obviously.
Step 2: 974169 * 974169
   GPT says:       948,000,000,000
   Correct answer: 949005240

## 7. When does it start failing?

Accuracy grouped by how many digits the correct answer has. This shows the point where the model stops being reliable.

In [8]:
from collections import defaultdict

by_size = defaultdict(lambda: [0, 0])          # digits -> [correct, total]
for r in results:
    for s in r["steps"]:
        bucket = (s["digits"] - 1) // 5 * 5 + 1  # 1-5, 6-10, 11-15 ...
        by_size[bucket][0] += s["right"]
        by_size[bucket][1] += 1

print(f"{'answer length':<16}{'correct':>10}")
for bucket in sorted(by_size):
    ok, total = by_size[bucket]
    print(f"{bucket:>2}-{bucket + 4:<2} digits    {ok:>4}/{total:<4} ({ok / total:.0%})")

print("\nExamples of wrong answers:")
for r in results:
    for s in r["steps"]:
        if not s["right"]:
            print(f"  {s['a']}² = {s['correct']}  |  GPT: {s['gpt']}  ({s['wrong_digits']} digits off)")

answer length      correct
 1-5  digits      22/22   (100%)
 6-10 digits       9/10   (90%)
11-15 digits       1/3    (33%)
16-20 digits       1/6    (17%)
21-25 digits       0/1    (0%)

Examples of wrong answers:
  4294967296² = 18446744073709551616  |  GPT: 18446744065119617024  (10 digits off)
  429981696² = 184884258895036416  |  GPT: 184987232174174976  (12 digits off)
  815730721² = 665416609183179841  |  GPT: 665560174747927841  (12 digits off)
  1874161² = 3512479453921  |  GPT: 3512329277475321  (15 digits off)
  96059601² = 9227446944279201  |  GPT: 923583764736747201  (17 digits off)
  15129² = 228886641  |  GPT: 2286142441  (8 digits off)
  228886641² = 52389094428262881  |  GPT: 52417467917417481  (13 digits off)
  974169² = 949005240561  |  GPT: 948000000000  (7 digits off)
  949005240561² = 900610946612241479594721  |  GPT: 900911927174174174174081  (16 digits off)


## 8. Download the output file

In [9]:
try:
    from google.colab import files
    files.download("assignment_3_output.txt")
except ImportError:
    print("Not in Colab: assignment_3_output.txt is in the current folder")